# ViT 이해하기

Vision Transformer 로 CIFAR-10 사진을 분류한다.

## 1. 런타임


In [ ]:
from io import BytesIO
from pathlib import Path

import torch
import torchvision
from IPython.display import display
from PIL import Image, ImageFont
from matplotlib import font_manager, pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from torch import nn
from torchvision import transforms
from torchvision.datasets import CIFAR10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))

!apt-get -qq update
!apt-get -qq install -y fonts-noto-cjk fonts-nanum


def hangul_font(size=14):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            font = ImageFont.truetype(path, size, **kw)
            try:
                font_manager.fontManager.addfont(path)
                plt.rcParams["font.family"] = font_manager.FontProperties(fname=path).get_name()
            except (OSError, ValueError, RuntimeError):
                pass
            print("hangul_font", path)
            return font
        except OSError:
            continue
    print("hangul_font missing")
    return None


hangul_font()
plt.rcParams["axes.unicode_minus"] = False


def show_img(img, width=900):
    img = img.convert("RGB")
    w, h = img.size
    if w > width:
        img = img.resize((width, int(h * width / w)), Image.BILINEAR)
    display(img)


def draw_blocks(title, nodes, arrows=(), note="", figsize=(12.2, 3.0)):
    fig, ax = plt.subplots(figsize=figsize)
    boxes = {}
    for n in nodes:
        bw, bh = n.get("bw", 1.85), n.get("bh", 1.12)
        cx, cy = n["cx"], n["cy"]
        ax.add_patch(
            FancyBboxPatch(
                (cx - bw / 2, cy - bh / 2),
                bw,
                bh,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                facecolor=n.get("color", "#ececec"),
                edgecolor="#222",
                linewidth=1.1,
            )
        )
        ax.text(cx, cy + 0.20, n["title"], ha="center", va="center", fontsize=8.2, fontweight="bold")
        ax.text(cx, cy - 0.22, n["sub"], ha="center", va="center", fontsize=7.2, color="#333")
        boxes[n["key"]] = (cx, cy, bw, bh)
    xs, ys = [], []
    for cx, cy, bw, bh in boxes.values():
        xs += [cx - bw / 2, cx + bw / 2]
        ys += [cy - bh / 2, cy + bh / 2]
    ax.set_xlim(min(xs) - 0.35, max(xs) + 0.35)
    ax.set_ylim(min(ys) - 0.75, max(ys) + 0.9)
    ax.axis("off")
    ax.set_title(title, fontsize=13, pad=8)
    for item in arrows:
        src, dst = item[0], item[1]
        text = item[2] if len(item) > 2 else ""
        style = item[3] if len(item) > 3 else "plain"
        x1, y1, w1, h1 = boxes[src]
        x2, y2, w2, h2 = boxes[dst]
        skip = style == "skip"
        color = "#c05621" if skip else "#333"
        dx = x2 - x1
        dy = y2 - y1
        if skip:
            patch = FancyArrowPatch(
                (x1, y1 + h1 / 2),
                (x2, y2 + h2 / 2),
                arrowstyle="-|>",
                mutation_scale=10,
                lw=1.25,
                color=color,
                connectionstyle="arc3,rad=0.42",
                linestyle=(0, (4, 3)),
            )
        elif abs(dx) >= abs(dy):
            x_s = x1 + w1 / 2 + 0.02 if dx > 0 else x1 - w1 / 2 - 0.02
            x_e = x2 - w2 / 2 - 0.02 if dx > 0 else x2 + w2 / 2 + 0.02
            patch = FancyArrowPatch((x_s, y1), (x_e, y2), arrowstyle="-|>", mutation_scale=10, lw=1.25, color=color)
        else:
            y_s = y1 - h1 / 2 - 0.02 if dy < 0 else y1 + h1 / 2 + 0.02
            y_e = y2 + h2 / 2 + 0.02 if dy < 0 else y2 - h2 / 2 - 0.02
            patch = FancyArrowPatch((x1, y_s), (x2, y_e), arrowstyle="-|>", mutation_scale=10, lw=1.25, color=color)
        ax.add_patch(patch)
        if text:
            ax.text((x1 + x2) / 2, max(y1, y2) + (0.78 if skip else 0.62), text, ha="center", fontsize=7, color="#444")
    if note:
        ax.text((min(xs) + max(xs)) / 2, min(ys) - 0.45, note, ha="center", fontsize=8, color="#333")
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    show_img(Image.open(buf), width=980)


이미지를 다루는 여러가지 방법론이 있는데 가장 대중적인 방법 중 하나는 Vision Transformer (ViT) 를 이용하는 방법입니다.
ViT는 이미지를 patch 토큰으로 잘라 한 줄로 넣은 뒤 트랜스포머 인코더가 토큰 사이의 관계를 봅니다.
분류를 하려면 앞에 CLS 토큰을 붙입니다. 인코더를 지난 CLS 줄을 Linear 에 넣어 클래스 점수를 냅니다.

CIFAR-10 은 32×32 색 사진 열 종류입니다. patch 한 변은 4 이라서 토큰은 64개입니다.


## 2. 데이터


CIFAR-10 학습 사진 5만 장 가운데 4만 5천 장으로 학습하고 5천 장으로 검증합니다. 시험 1만 장은 학습이 끝난 뒤에 봅니다.

학습 쪽에는 좌우 뒤집기와 crop 을 넣습니다.


In [ ]:
IMG_SIZE = 32
PATCH = 4
N_PATCHES = (IMG_SIZE // PATCH) ** 2
EMBED_DIM = 256
HIDDEN_DIM = 512
N_HEADS = 8
N_LAYERS = 6
NUM_CLASSES = 10
DROPOUT = 0.2
BATCH = 128
LR = 3e-4
EPOCHS = 12
CIFAR_MEAN = [0.49139968, 0.48215841, 0.44653091]
CIFAR_STD = [0.24703223, 0.24348513, 0.26158784]
CLASSES = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)
DATA_ROOT = Path("/content/cifar10") if Path("/content").exists() else Path.cwd() / "cifar10"

train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomResizedCrop((IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), ratio=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)

train_dataset = CIFAR10(root=DATA_ROOT, train=True, transform=train_transform, download=True)
val_dataset = CIFAR10(root=DATA_ROOT, train=True, transform=eval_transform, download=True)
test_set = CIFAR10(root=DATA_ROOT, train=False, transform=eval_transform, download=True)

gen = torch.Generator().manual_seed(42)
train_set, _ = torch.utils.data.random_split(train_dataset, [45000, 5000], generator=gen)
gen = torch.Generator().manual_seed(42)
_, val_set = torch.utils.data.random_split(val_dataset, [45000, 5000], generator=gen)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=2, pin_memory=DEVICE.type == "cuda"
)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH, shuffle=False, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH, shuffle=False, num_workers=2)

print("train", len(train_set), "val", len(val_set), "test", len(test_set))
print("patch", PATCH, "토큰", N_PATCHES, "클래스", NUM_CLASSES)

preview = torch.stack([val_set[i][0] for i in range(8)])
grid = torchvision.utils.make_grid(preview, nrow=8, normalize=True, pad_value=0.9)
show_img(transforms.functional.to_pil_image(grid), width=980)


## 3. 모델

아래는 전체를 한 줄로 본 그림입니다. 이어서 클래스마다 코드와 구조를 봅니다.


In [ ]:
draw_blocks(
    "ViT  (B, 3, 32, 32) → logits (B, 10)",
    [
        {"key": "in", "cx": 1.0, "cy": 1.3, "title": "image", "sub": "(B, 3, 32, 32)", "color": "#ececec", "bw": 1.7},
        {"key": "pt", "cx": 2.95, "cy": 1.3, "title": "img_to_patch", "sub": "4×4, N=64", "color": "#9ec5e8", "bw": 1.85},
        {"key": "lin", "cx": 4.95, "cy": 1.3, "title": "Linear", "sub": "48 → 256", "color": "#b7d4a5", "bw": 1.7},
        {"key": "cls", "cx": 6.9, "cy": 1.3, "title": "+ CLS, pos", "sub": "(B, 65, 256)", "color": "#f4d58d", "bw": 1.8},
        {"key": "enc", "cx": 8.9, "cy": 1.3, "title": "Encoder ×6", "sub": "attn + MLP", "color": "#c5b0d5", "bw": 1.8},
        {"key": "out", "cx": 10.8, "cy": 1.3, "title": "CLS Linear", "sub": "(B, 10)", "color": "#ececec", "bw": 1.65},
    ],
    [("in", "pt"), ("pt", "lin"), ("lin", "cls"), ("cls", "enc"), ("enc", "out")],
    figsize=(12.4, 2.8),
)


### img_to_patch

`img_to_patch`는 사진을 겹치지 않는 patch 로 자릅니다. 32×32 를 4×4 로 자르면 가로 8칸 세로 8칸이라 토큰은 64개입니다. 각 patch 는 RGB 를 펼친 48숫자입니다.


In [ ]:
def img_to_patch(x, patch_size, flatten_channels=True):
    # x: (B, C, H, W)
    batch, channels, height, width = x.shape
    x = x.reshape(batch, channels, height // patch_size, patch_size, width // patch_size, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)
    x = x.flatten(1, 2)
    if flatten_channels:
        x = x.flatten(2, 4)
    return x


draw_blocks(
    "img_to_patch  겹치지 않는 4×4 patch 64개",
    [
        {"key": "im", "cx": 1.3, "cy": 1.3, "title": "image", "sub": "(B, 3, 32, 32)", "color": "#ececec", "bw": 1.9},
        {"key": "cut", "cx": 3.7, "cy": 1.3, "title": "reshape", "sub": "8×8 격자", "color": "#9ec5e8", "bw": 1.8},
        {"key": "tok", "cx": 6.2, "cy": 1.3, "title": "tokens", "sub": "(B, 64, 3, 4, 4)", "color": "#c5b0d5", "bw": 2.0},
        {"key": "flat", "cx": 8.8, "cy": 1.3, "title": "flatten", "sub": "(B, 64, 48)", "color": "#f4d58d", "bw": 1.8},
    ],
    [("im", "cut"), ("cut", "tok"), ("tok", "flat")],
    note="펼친 RGB patch는 4×4×3=48 숫자입니다.",
)

x = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE)
print("image", tuple(x.shape), "tokens", tuple(img_to_patch(x, PATCH).shape))

sample = torch.stack([val_set[i][0] for i in range(4)])
patches = img_to_patch(sample, PATCH, flatten_channels=False)
fig, axes = plt.subplots(patches.size(0), 1, figsize=(14, 3))
fig.suptitle("Images as input sequences of patches")
for i in range(patches.size(0)):
    grid = torchvision.utils.make_grid(patches[i], nrow=N_PATCHES, normalize=True, pad_value=0.9)
    axes[i].imshow(grid.permute(1, 2, 0).cpu())
    axes[i].axis("off")
buf = BytesIO()
fig.savefig(buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
plt.close(fig)
buf.seek(0)
show_img(Image.open(buf), width=980)


### AttentionBlock

`AttentionBlock`은 Pre-Norm 입니다. LayerNorm 을 먼저 한 뒤 `nn.MultiheadAttention` 과 MLP 를 쌓고 잔차를 더합니다. 이 모듈은 시퀀스 칸이 앞에 오므로 텐서는 `(토큰, 배치, D)` 입니다.


In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_heads, dropout=0.0):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.linear = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        inp_x = self.layer_norm_1(x)
        x = x + self.attn(inp_x, inp_x, inp_x)[0]
        x = x + self.linear(self.layer_norm_2(x))
        return x


draw_blocks(
    "AttentionBlock  Pre-Norm, dashed = residual",
    [
        {"key": "x", "cx": 0.95, "cy": 1.35, "title": "x", "sub": "(65, B, 256)", "color": "#ececec", "bw": 1.5},
        {"key": "ln1", "cx": 2.6, "cy": 1.35, "title": "LayerNorm", "sub": "D=256", "color": "#9ec5e8", "bw": 1.5},
        {"key": "attn", "cx": 4.4, "cy": 1.35, "title": "attention", "sub": "8 heads", "color": "#c5b0d5", "bw": 1.6},
        {"key": "add1", "cx": 6.15, "cy": 1.35, "title": "+", "sub": "residual", "color": "#f4d58d", "bw": 1.2},
        {"key": "ln2", "cx": 7.7, "cy": 1.35, "title": "LayerNorm", "sub": "D=256", "color": "#9ec5e8", "bw": 1.5},
        {"key": "mlp", "cx": 9.45, "cy": 1.35, "title": "MLP", "sub": "256→512→256", "color": "#b7d4a5", "bw": 1.7},
        {"key": "add2", "cx": 11.25, "cy": 1.35, "title": "+", "sub": "residual", "color": "#f4d58d", "bw": 1.2},
    ],
    [
        ("x", "ln1"),
        ("ln1", "attn"),
        ("attn", "add1"),
        ("add1", "ln2"),
        ("ln2", "mlp"),
        ("mlp", "add2"),
        ("x", "add1", "", "skip"),
        ("add1", "add2", "", "skip"),
    ],
    figsize=(13.0, 3.2),
)

block = AttentionBlock(EMBED_DIM, HIDDEN_DIM, N_HEADS, DROPOUT)
x = torch.zeros(N_PATCHES + 1, 2, EMBED_DIM)
print("x", tuple(x.shape), "out", tuple(block(x).shape))


### VisionTransformer

Linear 가 각 patch 를 embed_dim 으로 보냅니다. 앞에 CLS 토큰을 붙이고 positional embedding 을 더합니다. 인코더를 지난 뒤 CLS 줄만 클래스 점수를 냅니다.


In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        embed_dim,
        hidden_dim,
        num_channels,
        num_heads,
        num_layers,
        num_classes,
        patch_size,
        num_patches,
        dropout=0.0,
    ):
        super().__init__()
        self.patch_size = patch_size
        self.input_layer = nn.Linear(num_channels * (patch_size**2), embed_dim)
        self.transformer = nn.Sequential(
            *[AttentionBlock(embed_dim, hidden_dim, num_heads, dropout=dropout) for _ in range(num_layers)]
        )
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes),
        )
        self.dropout = nn.Dropout(dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, 1 + num_patches, embed_dim))

    def forward(self, x):
        x = img_to_patch(x, self.patch_size)
        batch, n_tokens, _ = x.shape
        x = self.input_layer(x)
        cls_token = self.cls_token.repeat(batch, 1, 1)
        x = torch.cat([cls_token, x], dim=1)
        x = x + self.pos_embedding[:, : n_tokens + 1]
        x = self.dropout(x)
        x = x.transpose(0, 1)
        x = self.transformer(x)
        cls = x[0]
        return self.mlp_head(cls)


draw_blocks(
    "VisionTransformer head  인코더 출력의 맨 앞(CLS)만 클래스를 냅니다",
    [
        {"key": "enc", "cx": 2.2, "cy": 1.35, "title": "encoder out", "sub": "(65, B, 256)", "color": "#c5b0d5", "bw": 2.1},
        {"key": "cls", "cx": 5.2, "cy": 1.35, "title": "x[0]", "sub": "CLS (B, 256)", "color": "#f4d58d", "bw": 2.0},
        {"key": "lin", "cx": 8.1, "cy": 1.35, "title": "LayerNorm+Linear", "sub": "256 → 10", "color": "#e8a090", "bw": 2.1},
        {"key": "out", "cx": 10.8, "cy": 1.35, "title": "logits", "sub": "(B, 10)", "color": "#ececec", "bw": 1.7},
    ],
    [("enc", "cls"), ("cls", "lin"), ("lin", "out")],
    figsize=(12.4, 2.8),
)

model = VisionTransformer(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_channels=3,
    num_heads=N_HEADS,
    num_layers=N_LAYERS,
    num_classes=NUM_CLASSES,
    patch_size=PATCH,
    num_patches=N_PATCHES,
    dropout=DROPOUT,
)
x = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE)
print("image", tuple(x.shape), "logits", tuple(model(x).shape))
print("파라미터", f"{sum(p.numel() for p in model.parameters()):,}")


## 4. 학습

손실은 교차 엔트로피입니다. 최적화는 AdamW 입니다. 아래 셀이 학습 루프입니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer, train):
    model.train(train)
    total_loss, n_ok, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        pred = logits.argmax(1)
        bs = y.size(0)
        total_loss += loss.item() * bs
        n_ok += int((pred == y).sum().item())
        n += bs
    return total_loss / max(n, 1), n_ok / max(n, 1)


model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[8, 10], gamma=0.1)
best_acc = -1.0
best_path = Path("vit_cifar10_best.pt")
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, True)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer, False)
    scheduler.step()
    print(
        f"epoch {epoch:02d}  train_loss {tr_loss:.4f}  train_acc {tr_acc:.4f}  "
        f"val_loss {va_loss:.4f}  val_acc {va_acc:.4f}"
    )
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), best_path)
print("best val_acc", round(best_acc, 4), "saved", best_path)


## 5. 확인

검증에서 고른 가중치로 시험 사진을 봅니다.


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
te_loss, te_acc = run_epoch(model, test_loader, criterion, None, False)
print("test_loss", round(te_loss, 4), "test_acc", round(te_acc, 4))

model.eval()
x, y = next(iter(test_loader))
x = x[:16].to(DEVICE)
with torch.no_grad():
    pred = model(x).argmax(1).cpu()
grid = torchvision.utils.make_grid(x.cpu(), nrow=8, normalize=True, pad_value=0.9)
show_img(transforms.functional.to_pil_image(grid), width=980)
print("정답", [CLASSES[int(v)] for v in y[:16]])
print("예측", [CLASSES[int(v)] for v in pred])
